# Packages to Import

In [ ]:
from __future__ import print_function, unicode_literals, absolute_import, division
import sys
import numpy as np
import matplotlib
matplotlib.rcParams["image.interpolation"] = 'none'
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from glob import glob
from tifffile import imread
from csbdeep.utils import Path, normalize
from csbdeep.io import save_tiff_imagej_compatible

from stardist import random_label_cmap, _draw_polygons, export_imagej_rois
from stardist.models import StarDist2D

np.random.seed(6)
lbl_cmap = random_label_cmap()

from tqdm import tqdm
import napari
import skimage as sk
import pandas as pd
import os
from aicspylibczi import CziFile
from Fix_File_Names import replace_extension, replace_spaces_with_underscores_recursive, fix_double_underscores_recursive, fix_naming

In [ ]:
#replace tiff extension with tif
replace_extension(r'Image_Data\Oct_2025_300mm_exp\50_mM\Z_slices')
replace_extension(r'Image_Data\Oct_2025_300mm_exp\300_mM_1_hr\Z_slices')
replace_extension(r'Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Z_slices')

In [ ]:
replace_spaces_with_underscores_recursive(r'C:\OIC-181\300_mm_90_min')

In [ ]:
fix_naming(r'C:\OIC-181\300_mm_90_min')

## StarDist Prediction and Object Analysis
This section is specifically for detecting and quantifying the ring-like structures observed in the oocytes

### Prepping image slices to be analyzed

In [ ]:
#na_50_files = sorted(glob(r'C:\OIC-181\50_mM\*.czi'))
#na_300_90_min_files = sorted(glob(r'C:\OIC-181\300_mm_90_min\*.czi'))
na_300_6_hr_files = sorted(glob(r'C:\OIC-181\300_mm_6_hr\*.czi'))

In [ ]:
len(na_50_files)

In [ ]:
# na_50_files_names = list(map(os.path.basename,na_50_files))
# na_300_90_min_files_names = list(map(os.path.basename,na_300_90_min_files))
na_300_6_hr_files_names = list(map(os.path.basename,na_300_6_hr_files))

In [ ]:
print(na_300_90_min_files_names)

In [ ]:
# na_50_czi = list(map(CziFile,na_50_files))
# na_300_90_min_czi = list(map(CziFile,na_300_90_min_files))
na_300_6_hr_czi = list(map(CziFile,na_300_6_hr_files))

In [ ]:
#reference names
# na_50_pos_z_files= sorted(glob(r'C:\OIC-181\50_mm\50_mm_z_slices_with_ROI\*.czi'))
# na_300_90_min_pos_z_files = sorted(glob(r'C:\OIC-181\300_mm_90_min\300_mm_90_min_z_slices_with_ROI\*.czi'))
na_300_6_hr_pos_z_files = sorted(glob(r'C:\OIC-181\300_mm_6_hr\300_mm_6_hr_z_slices_with_ROI\*.czi'))

In [ ]:
len(na_300_90_min_pos_z_files)

In [ ]:
#get base name of all images
# na_50_pos_z_names = list(map(os.path.basename,na_50_pos_z_files))
# na_300_90_min_pos_z_names = list(map(os.path.basename,na_300_90_min_pos_z_files))
na_300_6_hr_pos_z_names = list(map(os.path.basename,na_300_6_hr_pos_z_files))

Get list of image positions and slices to analyze using naming pattern from extracted slices with ROIs, created tuples of the scenes and z slices to use for extracting the data subset from the original data.

In [ ]:
#lists of positions and z for analysis
import re
from collections import defaultdict

def extract_and_group_scenes(file_list):
    """
    Extract (pos, z) tuples from filenames and group by source image.
    
    Args:
        file_list: List of filenames with pattern like 'airyscan_6_pos1_z_8.czi'
    
    Returns:
        List of lists, where each inner list contains (pos, z) tuples grouped by source image
    """
    # Dictionary to group tuples by source image prefix
    grouped = defaultdict(list)
    
    for filename in file_list:
        # Extract the source image prefix (everything before _pos)
        source_match = re.match(r'^(.+?)_pos', filename)
        source_key = source_match.group(1) if source_match else None
        
        # Extract pos number and z number
        if re.search(r'_pos(\d+)_z_(\d+).czi', filename):
            pos_match = re.search(r'_pos(\d+)', filename)
            z_match = re.search(r'_z_(\d+)', filename)
            
            if pos_match and z_match and source_key:
                pos = int(pos_match.group(1))
                z = int(z_match.group(1))
                grouped[source_key].append((pos, z))
        
        if re.search(r'_pos(\d+)_z_(\d+)_(\d+).czi', filename):
            pos_match = re.search(r'_pos(\d+)', filename)
            z_match = re.search(r'_z_(\d+)_(\d+)', filename)
        
            if pos_match and z_match and source_key:
                pos = int(pos_match.group(1))
                z = int(z_match.group(1))
                duplicate = int(z_match.group(2)) # Handle duplicates if present
                grouped[source_key].append((pos, z, duplicate))

    # Convert to list of lists, maintaining order
    result = list(grouped.values())
    return result

# Example usage:
file_list = [
    'airyscan_6_pos1_z_8_1.czi',
    'airyscan_6_pos3_z_38_2.czi',
    '29_airyscan_pos1_z_26.czi',
    '29_airyscan_pos2_z_17_2.czi'
]

# na_50_pos_z = extract_and_group_scenes(na_50_pos_z_names)
# na_300_1_pos_z = extract_and_group_scenes(na_300_1_pos_z_names)
# na_300_6_pos_z = extract_and_group_scenes(na_300_6_pos_z_names)


In [ ]:
test_list = extract_and_group_scenes(file_list)
print(test_list)

In [ ]:
# na_50_pos_z = extract_and_group_scenes(na_50_pos_z_names)
# na_300_90_min_pos_z = extract_and_group_scenes(na_300_90_min_pos_z_names)
na_300_6_hr_pos_z = extract_and_group_scenes(na_300_6_hr_pos_z_names)

In [ ]:
len(na_300_6_hr_pos_z_names)

In [ ]:
print(na_50_pos_z)

Found additional inconsistent naming schemes, names of data subset for image 5 from the 300 mM Na dataset had an added underscore between "pos" and the number. Had to fix this for the data extraction to run smoothly.

In [ ]:
print("na_300_90_min_pos_z_names:")
print(na_300_90_min_pos_z_names)
print("\nna_300_6_hr_pos_z_names:")
print(na_300_6_hr_pos_z_names)

In [ ]:
#where to save the extracted subset
# na_50_slices_path = r'C:\OIC-181\50_mm\Z_slices'
# na_300_90_min_slices_path = r'C:\OIC-181\300_mm_90_min\Z_slices'
na_300_6_hr_slices_path = r'C:\OIC-181\300_mm_6_hr\Z_slices'

Selected the respective images from the original czi files and saved as individual tiffs

In [ ]:
for img, pos, name in zip(na_50_czi,na_50_pos_z,na_50_files_names):
    path = na_50_slices_path
    for scene, z, dup in pos:
        slice = np.squeeze(img.read_image(S=scene-1,Z=z-1)[0])
        sk.io.imsave(os.path.join(path,name[:-4]+'_pos'+str(scene)+'_z_'+str(z)+'_'+str(dup)+'.tif'),slice,check_contrast=False)

In [ ]:
for img, pos, name in zip(na_300_90_min_czi,na_300_90_min_pos_z,na_300_90_min_files_names):
    path = na_300_90_min_slices_path
    for scene, z, dup in pos:
        slice = np.squeeze(img.read_image(S=scene-1,Z=z-1)[0])
        sk.io.imsave(os.path.join(path,name[:-4]+'_pos'+str(scene)+'_z_'+str(z)+'_'+str(dup)+'.tif'),slice,check_contrast=False)

In [ ]:
for img, pos, name in zip(na_300_6_hr_czi,na_300_6_hr_pos_z,na_300_6_hr_files_names):
    path = na_300_6_hr_slices_path
    for scene, z, dup in pos:
        if os.path.join(path,name[:-4]+'_pos'+str(scene)+'_z_'+str(z)+'_'+str(dup)+'.tif') not in glob(os.path.join(path,'*.tif')):
            slice = np.squeeze(img.read_image(S=scene-1,Z=z-1)[0])
            sk.io.imsave(os.path.join(path,name[:-4]+'_pos'+str(scene)+'_z_'+str(z)+'_'+str(dup)+'.tif'),slice,check_contrast=False)
        else:
            continue

Double check if there are any other files names that are not matching the naming scheme

In [ ]:
def extract_and_group_scenes_debug(file_list):
    """Debug version to see what's being extracted"""
    grouped = defaultdict(list)
    non_matching = []
    
    for filename in file_list:
        source_match = re.match(r'^([^_]*_[^_]*)_pos', filename)
        source_key = source_match.group(1) if source_match else None
        
        pos_match = re.search(r'_pos(\d+)_', filename)
        z_match = re.search(r'_z_(\d+)', filename)
        
        print(f"File: {filename}")
        print(f"  source_key: {source_key}, pos: {pos_match.group(1) if pos_match else None}, z: {z_match.group(1) if z_match else None}")
        
        if pos_match and z_match and source_key:
            pos = int(pos_match.group(1))
            z = int(z_match.group(1))
            grouped[source_key].append((pos, z))
        else:
            non_matching.append({
                'filename': filename,
                'source_key': source_key,
                'has_pos': pos_match is not None,
                'has_z': z_match is not None
            })
    
    result = list(grouped.values())
    print(f"\nFinal grouped result: {result}")
    
    if non_matching:
        print(f"\n⚠️  {len(non_matching)} files did NOT match the regex pattern:")
        for item in non_matching:
            print(f"  - {item['filename']}")
            print(f"    source_key: {item['source_key']}, has_pos: {item['has_pos']}, has_z: {item['has_z']}")
    else:
        print("\n✓ All files matched the regex pattern!")
    
    return result

# Test it
print("=" * 50)
print("Debugging na_300_1_pos_z_names:")
print("=" * 50)
na_300_1_pos_z_debug = extract_and_group_scenes_debug(na_300_1_pos_z_names)

In [ ]:
extract_and_group_scenes_debug(na_50_pos_z_names)

### Trained Custom StarDist Model to Detect Ring Structures

Notebook outlining stardist training is in [Transfer_learning_StarDist](/Transfer_learning_StarDist.ipynb)

Run custom StarDist model on data subset extracted above

In [ ]:
na_50_slices_files = sorted(glob(r'Image_Data\Nov_2025_300mm_exp\50_mm\Z_slices\*.tif'))
na_300_1_slices_files = sorted(glob(r'Image_Data\Nov_2025_300mm_exp\300_mm_90_min\Z_slices\*.tif'))
#na_300_6_slices_files = sorted(glob(r'Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Z_slices\*.tif'))

In [ ]:
na_50_slices_names = list(map(os.path.basename,na_50_slices_files))
na_300_1_slices_names = list(map(os.path.basename,na_300_1_slices_files))
#na_300_6_slices_names = list(map(os.path.basename,na_300_6_slices_files))

In [ ]:
na_50_slices = list(map(imread,na_50_slices_files))
na_300_1_slices = list(map(imread,na_300_1_slices_files))
#na_300_6_slices = list(map(imread,na_300_6_slices_files))

In [ ]:
#check image dimensions for image normalization
n_channel = 1 if na_50_slices[0].ndim == 2 else X[0].shape[-1]
axis_norm = (0,1)
if n_channel > 1:
    print("Normalizing image channels %s." % ('jointly' if axis_norm is None or 2 in axis_norm else 'independently'))

Load in custom trained model and then run on all images to get objects

In [ ]:
#load in model
model = StarDist2D(None, name='2D_versatile_fluo_Rings', basedir='models')

In [ ]:
na_50_slices_norm = [normalize(x, 1,99.8, axis=axis_norm) for x in na_50_slices]
na_300_1_slices_norm = [normalize(x, 1,99.8, axis=axis_norm) for x in na_300_1_slices]
#na_300_6_slices_norm = [normalize(x, 1,99.8, axis=axis_norm) for x in na_300_6_slices]


In [ ]:
#Predict labels on full images, mask the labels with the manually created masks to remove background objects
na_50_slices_labels = [model.predict_instances(x)[0] for x in tqdm(na_50_slices_norm)]
na_300_1_slices_labels = [model.predict_instances(x)[0] for x in tqdm(na_300_1_slices_norm)]
#na_300_6_slices_labels = [model.predict_instances(x)[0] for x in tqdm(na_300_6_slices_norm)]

Check labels to see how model performed

In [ ]:
viewer = napari.view_image(na_300_1_slices[15])
viewer.add_labels(na_300_1_slices_labels[15])

In [ ]:
na_50_path = r'Image_Data\Nov_2025_300mm_exp\50_mm\Ring_masks'
na_300_1_path = r'Image_Data\Nov_2025_300mm_exp\300_mm_90_min\Ring_masks'
#na_300_6_path = r'Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Ring_masks'

In [ ]:
#save the prediction masks
for pred, name in zip(na_50_slices_labels,na_50_slices_names):
    path = na_50_path
    sk.io.imsave(os.path.join(path,"Ring_masks_"+name[:-4]+'.tif'),pred,check_contrast=False)

for pred, name in zip(na_300_1_slices_labels,na_300_1_slices_names):
    path = na_300_1_path
    sk.io.imsave(os.path.join(path,"Ring_masks_"+name[:-4]+'.tif'),pred,check_contrast=False)

# for pred, name in zip(na_300_6_slices_labels,na_300_6_slices_names):
#     path = na_300_6_path
#     sk.io.imsave(os.path.join(path,name[:-4]+'_ring_masks.tif'),pred,check_contrast=False)

In [ ]:
#read in oocyte masks to get object measurements for specific cells
na_50_oocyte_masks_files = sorted(glob(r'Image_Data\Nov_2025_300mm_exp\50_mm\Oocyte_masks\*.tif'))
na_300_1_oocyte_masks_files = sorted(glob(r'Image_Data\Nov_2025_300mm_exp\300_mm_90_min\Oocyte_masks\*.tif'))
#na_300_6_oocyte_masks_files = sorted(glob(r'Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Oocyte_masks\*.tif'))

#read in StarDist segmentation
na_50_SD_seg_files = sorted(glob(r'Image_Data\Nov_2025_300mm_exp\50_mm\Ring_masks\*.tif'))
na_300_1_SD_seg_files = sorted(glob(r'Image_Data\Nov_2025_300mm_exp\300_mm_90_min\Ring_masks\*.tif'))
# na_300_6_SD_seg_files = sorted(glob(r'Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Ring_masks\*.tif'))



In [ ]:
# Extract base names for sorting consistency due to added suffix
def get_base_name(filepath):
    """Extract base name without condition-specific suffix"""
    basename = os.path.basename(filepath)
    # Remove all known suffixes
    base = basename
    prefixes = ['Oocyte_mask_', 'Ring_masks_']
    for prefix in prefixes:
        base = base.replace(prefix, '')
    return base

# Sort both lists by their base names and pair them
na_50_slices_files_sorted = sorted(na_50_slices_files, key=lambda x: get_base_name(x))
na_50_oocyte_masks_files_sorted = sorted(na_50_oocyte_masks_files, key=lambda x: get_base_name(x))
na_50_SD_seg_files_sorted = sorted(na_50_SD_seg_files, key=lambda x: get_base_name(x))

na_300_1_slices_files_sorted = sorted(na_300_1_slices_files, key=lambda x: get_base_name(x))
na_300_1_oocyte_masks_files_sorted = sorted(na_300_1_oocyte_masks_files, key=lambda x: get_base_name(x))
na_300_1_SD_seg_files_sorted = sorted(na_300_1_SD_seg_files, key=lambda x: get_base_name(x))

# na_300_6_slices_files_sorted = sorted(na_300_6_slices_files, key=lambda x: get_base_name(x))
# na_300_6_oocyte_masks_files_sorted = sorted(na_300_6_oocyte_masks_files, key=lambda x: get_base_name(x))
# na_300_6_SD_seg_files_sorted = sorted(na_300_6_SD_seg_files, key=lambda x: get_base_name(x))

# Verify they match
for slice_file, mask_file in zip(na_50_slices_files_sorted, na_50_oocyte_masks_files_sorted):
    slice_base = get_base_name(slice_file)
    mask_base = get_base_name(mask_file)
    assert slice_base == mask_base, f"Mismatch: {slice_base} != {mask_base}"

In [ ]:
na_300_1_basenames = [get_base_name(x) for x in na_300_1_slices_files_sorted]
na_300_oocyte_mask_names = [get_base_name(x) for x in na_300_1_oocyte_masks_files_sorted]

In [ ]:
missing_mask = [item for item in na_300_1_basenames if item not in na_300_oocyte_mask_names]
print(missing_mask)

In [ ]:
#Get newly sorted names and read in arrays with the same order across all files
na_50_slices_names = list(map(os.path.basename,na_50_slices_files_sorted))
na_300_1_slices_names = list(map(os.path.basename,na_300_1_slices_files_sorted))
#na_300_6_slices_names = list(map(os.path.basename,na_300_6_slices_files_sorted))

na_50_slices = list(map(imread,na_50_slices_files_sorted))
na_300_1_slices = list(map(imread,na_300_1_slices_files_sorted))
#na_300_6_slices = list(map(imread,na_300_6_slices_files_sorted))

na_50_oocyte_masks_names = list(map(os.path.basename,na_50_oocyte_masks_files_sorted))
na_300_1_oocyte_masks_names = list(map(os.path.basename,na_300_1_oocyte_masks_files_sorted))
#na_300_6_oocyte_masks_names = list(map(os.path.basename,na_300_6_oocyte_masks_files_sorted))

na_50_oocyte_masks = list(map(sk.io.imread,na_50_oocyte_masks_files_sorted))
na_300_1_oocyte_masks = list(map(sk.io.imread,na_300_1_oocyte_masks_files_sorted))
#na_300_6_oocyte_masks = list(map(sk.io.imread,na_300_6_oocyte_masks_files_sorted))

na_50_SD_seg_names = list(map(os.path.basename,na_50_SD_seg_files_sorted))
na_300_1_SD_seg_names = list(map(os.path.basename,na_300_1_SD_seg_files_sorted))
#na_300_6_SD_seg_names = list(map(os.path.basename,na_300_6_SD_seg_files_sorted))

na_50_SD_seg = list(map(sk.io.imread,na_50_SD_seg_files_sorted))
na_300_1_SD_seg= list(map(sk.io.imread,na_300_1_SD_seg_files_sorted))
#na_300_6_SD_seg = list(map(sk.io.imread,na_300_6_SD_seg_files_sorted))


In [ ]:
merged_slices_50 = np.stack(na_50_slices)
merged_slices_300_1 = np.stack(na_300_1_slices)
merged_slices_300_6 = np.stack(na_300_6_slices)

merged_seg_50 = np.stack(na_50_SD_seg)
merged_seg_300_1 = np.stack(na_300_1_SD_seg)
merged_seg_300_6 = np.stack(na_300_6_SD_seg)

In [ ]:
merged_masks_50 = np.stack(na_50_oocyte_masks)
merged_masks_300_1 = np.stack(na_300_1_oocyte_masks)
merged_masks_300_6 = np.stack(na_300_6_oocyte_masks)

In [ ]:
viewer_50 = napari.view_image(merged_slices_50)
viewer_50.add_labels(merged_seg_50)
viewer_50.add_labels(merged_masks_50)


In [ ]:
print(na_50_slices_names[11],na_50_oocyte_masks_names[11],na_50_SD_seg_names[11])

In [ ]:
na_50_oocyte_masked_labels = [na_50_oocyte_masks[i]*na_50_SD_seg[i] for i in range(len(na_50_SD_seg))]
na_300_1_oocyte_masked_labels = [na_300_1_oocyte_masks[i]*na_300_1_SD_seg[i] for i in range(len(na_300_1_SD_seg))]
#na_300_6_oocyte_masked_labels = [na_300_6_oocyte_masks[i]*na_300_6_SD_seg[i] for i in range(len(na_300_6_SD_seg))]

Get measurements of all identified objects in oocytes

In [ ]:
pixel_size = [0.04,0.04]
props = ('label','axis_major_length','axis_minor_length','eccentricity','perimeter','area')

In [ ]:
def get_measurements(masks, names, props, pixel_size):
    """
    Get measurements from masks and concatenate all dataframes into one.
    
    Args:
        masks: List of mask images
        imgs: List of image arrays
        names: List of image names
        props: Properties to measure
        pixel_size: Pixel size for spacing
    
    Returns:
        Single concatenated dataframe with all measurements
    """
    dfs = []
    for mask, name in zip(masks, names):
        df = sk.measure.regionprops_table(mask, properties=props, spacing=pixel_size)
        df = pd.DataFrame.from_dict(df)
        df['image_name'] = name[:-4]
        dfs.append(df)
    
    # Concatenate all dataframes into one
    merged_df = pd.concat(dfs, ignore_index=True)
    # Move 'image_name' column to the front
    col = merged_df.pop('image_name')
    merged_df.insert(0, col.name, col)
    return merged_df

In [ ]:
def circularity_and_aspect_ratio(df):
    """
    Calculate the circularity and aspect ratio of all objects. 
    Requires that the perimeter, area, and major and minor axes have already been calculated.
    
    Args:
        df: data frame that contains the required measurements
    
    Returns:
        Single concatenated dataframe with all measurements
    """
    circularity_list = []
    aspect_ratio_list = []
    perimeters = np.asarray(df['perimeter']).astype(np.float64)
    areas = np.asarray(df['area']).astype(np.float64)
    min_diam = np.asarray(df['axis_minor_length']).astype(np.float64)
    max_diam = np.asarray(df['axis_major_length']).astype(np.float64)
    for c in range(len(perimeters)):
        circ = (4*np.pi*areas[c])/(perimeters[c]**2)
        circularity_list.append(circ)
        ar = max_diam[c]/min_diam[c]
        aspect_ratio_list.append(ar)
    aspect_ratios = pd.Series(aspect_ratio_list,name='aspect_ratio')
    circularities = pd.Series(circularity_list,name='circularity')
    merged_df = pd.concat([df,circularities,aspect_ratios],axis=1)
    return merged_df


In [ ]:
def save(save_path, file_names, masked_SD_seg, merged_df, condition):
    """
   Save the dataframes to the correct locations
    
    Args:
        save_path: path to save location
        file_names: list of names
        masked_SD_seg: segmentation masks from StarDist masked to be specific for each oocyte
        merged_df: data frame of the measurements calculated from get_measurements and circularity_and_aspect_ratio functions
        condition: the experimental condition for the data
    
    Returns:
        Single concatenated dataframe with all measurements
    """
    dataframe_path = os.path.join(save_path,'Measurements')
    merged_df.to_csv(os.path.join(dataframe_path,'measurements_'+condition+'.csv'))

    masked_SD_seg_path = os.path.join(save_path,'Masked_ring_masks')
    for mask,name in zip(masked_SD_seg,file_names):
        sk.io.imsave(os.path.join(masked_SD_seg_path,'Oocyte_masked_rings_'+name[:-4]+'.tif'),mask,check_contrast=False)
    

In [ ]:
na_50_save_path = r'Image_Data\Nov_2025_300mm_exp\50_mm'
na_300_1_save_path = r'Image_Data\Nov_2025_300mm_exp\300_mm_90_min'
#na_300_6_save_path = r'Image_Data\Oct_2025_300mm_exp\300_mM_6_hr'

# save_paths = [na_50_save_path,na_300_1_save_path,na_300_6_save_path]
# conditions = ["50mM","300mM_1.5hr","300mM_6hr"]
# masked_SD_preds = [na_50_oocyte_masked_labels,na_300_1_oocyte_masked_labels,na_300_6_oocyte_masked_labels]
# file_names = [na_50_slices_names,na_300_1_slices_names,na_300_6_slices_names]

save_paths = [na_50_save_path,na_300_1_save_path]
conditions = ["50mM","300mM_90_min"]
masked_SD_preds = [na_50_oocyte_masked_labels,na_300_1_oocyte_masked_labels]
file_names = [na_50_slices_names,na_300_1_slices_names]

for save_path, condition, objects, names in zip(save_paths,conditions,masked_SD_preds,file_names):
    df = get_measurements(masks=objects,names=names, props=props,pixel_size=pixel_size)
    final_df = circularity_and_aspect_ratio(df)
    save(save_path=save_path,file_names=names,masked_SD_seg=objects,merged_df=final_df,condition=condition)

In [ ]:
#Save masked and cropped copies of the oocyte images for skeleton analysis
# images = [na_50_slices,na_300_1_slices,na_300_6_slices]
# oocyte_masks = [na_50_oocyte_masks,na_300_1_oocyte_masks,na_300_6_oocyte_masks]

images = [na_50_slices,na_300_1_slices]
oocyte_masks = [na_50_oocyte_masks,na_300_1_oocyte_masks]

for path, name_set , slice_set, mask_set in zip(save_paths,file_names,images,oocyte_masks):
    save_path = os.path.join(path,"Masked_cropped_images")
    for name,slice,mask in zip(name_set,slice_set,mask_set):
        props = sk.measure.regionprops(mask)
        minc,minr,maxc,maxr = props[0].bbox
        masked_img = mask*slice
        cropped_img = masked_img[minc:maxc,minr:maxr]
        sk.io.imsave(os.path.join(save_path,'Cropped_masked_'+name[:-4]+".tif"),cropped_img,check_contrast=False)

# Options for plotting and running a linear mixed effects model, not modified yet

In [ ]:
na_50_measurements = pd.read_csv(r"Image_Data\Oct_2025_300mm_exp\50_mM\Measurements\measurements_50mM.csv")
na_300_1_measurements = pd.read_csv(r"Image_Data\Oct_2025_300mm_exp\300_mM_1_hr\Measurements\measurements_300mM_1.5hr.csv")
na_300_6_measurements = pd.read_csv(r"Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Measurements\measurements_300mM_6hr.csv")

In [ ]:
na_50_measurements['condition'] = '50mM'
na_300_1_measurements['condition'] = '300mM_1.5hr'
na_300_6_measurements['condition'] = '300mM_6hr'
all_data = pd.concat([na_50_measurements,na_300_1_measurements,na_300_6_measurements],ignore_index=True)

In [ ]:
all_data.to_csv(r'Output\Oct_2025_300mM_exp\all_conditions_ring_structures_analysis.csv')

In [ ]:
area_50 = na_50_measurements['area']
area_300_1 = na_300_1_measurements['area']
area_300_6 = na_300_6_measurements['area']

ar_50 = na_50_measurements['aspect_ratio']
ar_300_1 = na_300_1_measurements['aspect_ratio']
ar_300_6 = na_300_6_measurements['aspect_ratio']

circ_50 = na_50_measurements['circularity']
circ_300_1 = na_300_1_measurements['circularity']
circ_300_6 = na_300_6_measurements['circularity']

# ar_50 = [df['aspect_ratio'] for df in na_50_measurements]
# ar_500 = [df['aspect_ratio'] for df in na_500_measurements]
# circ_50 = [df['circularity'] for df in na_50_measurements]
# circ_500 = [df['circularity'] for df in na_500_measurements]
# intensity_50 = [df['intensity_mean'] for df in na_50_measurements]
# intensity_500 = [df['intensity_mean'] for df in na_500_measurements]

In [ ]:
fig, ax = plt.subplots(3,3,figsize=(12,12),layout='constrained')
fig.suptitle('Mitochondrial Ring Measurement Distributions', fontsize=16,fontweight='bold')
# Calculate y-axis limits based on all data
y_min = min(area_50.min(), area_300_1.min(), area_300_6.min())
y_max = max(area_50.max(), area_300_1.max(), area_300_6.max())

# Area boxplots
ax[0,0].boxplot(area_50)
ax[0,0].set_title('Area 50mM Na')
ax[0,0].set_yscale('log', base=10)
ax[0,0].set_ylim(0.001,y_max+1)

ax[0,1].boxplot(area_300_1)
ax[0,1].set_title('Area 300mM Na 1.5hr')
ax[0,1].set_yscale('log', base=10)
ax[0,1].set_ylim(0.001,y_max+1)

ax[0,2].boxplot(area_300_6)
ax[0,2].set_title('Area 300mM Na 6hr')
ax[0,2].set_yscale('log', base=10)
ax[0,2].set_ylim(0.001,y_max+1)

#Aspect Ratio Boxplots
ax[1,0].boxplot(ar_50)
ax[1,0].set_title('Aspect Ratio 50mM Na')
ax[1,0].set_yscale('log', base=10)
ax[1,0].set_ylim(0,6)

ax[1,1].boxplot(ar_300_1)
ax[1,1].set_title('Aspect Ratio 300mM Na 1.5hr')
ax[1,1].set_yscale('log', base=10)
ax[1,1].set_ylim(0,6)

ax[1,2].boxplot(ar_300_6)
ax[1,2].set_title('Aspect Ratio 300mM Na 6hr')
ax[1,2].set_yscale('log', base=10)
ax[1,2].set_ylim(0,6)

#Circularity boxplots
ax[2,0].boxplot(circ_50)
ax[2,0].set_title('Circularity 50mM Na')
ax[2,0].set_yscale('log', base=10)
ax[2,0].set_ylim(0.6,1.2)

ax[2,1].boxplot(circ_300_1)
ax[2,1].set_title('Circularity 300mM Na 1.5hr')
ax[2,1].set_yscale('log', base=10)
ax[2,1].set_ylim(0.6,1.2)

ax[2,2].boxplot(circ_300_6)
ax[2,2].set_title('Circularity 300mM Na 6hr')
ax[2,2].set_yscale('log', base=10)
ax[2,2].set_ylim(0.6,1.2)
plt.show()

## Linear mixed effects model to compare areas

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os
from glob import glob
import matplotlib.pyplot as plt

In [ ]:
na_50_measurements = sorted(glob('Output/measurements/*_50mm_*.csv'))
na_500_measurements = sorted(glob('Output/measurements/*_500mm_*.csv'))

In [ ]:
na_50_measurements = list(map(pd.read_csv,na_50_measurements))
na_500_measurements = list(map(pd.read_csv,na_500_measurements))

In [ ]:
na_50_measurements[0].head()

Externally added in a column for the bio_rep and condition for setting up the mixed effects model

In [ ]:
area_df_na50 = [df.get(['area','bio_rep','condition']) for df in na_50_measurements]
area_df_na500 = [df.get(['area','bio_rep','condition']) for df in na_500_measurements]
area_df_na50_merged = pd.concat(area_df_na50).reset_index(drop=True)
area_df_na500_merged = pd.concat(area_df_na500).reset_index(drop=True)

In [ ]:
#verify concat worked
area_df_na50_merged.tail()

In [ ]:
dfs_merged = pd.concat([area_df_na50_merged,area_df_na500_merged]).reset_index(drop=True)

Testing square root and log normalization for better model performance

In [ ]:
dfs_merged['area_sqrt'] = np.sqrt(dfs_merged['area'])

In [ ]:
dfs_merged['area_log'] = np.log(dfs_merged['area'])

In [ ]:
#setting up and running mixed effects model
model = smf.mixedlm(
    "area_log ~ C(condition)",
    data=dfs_merged,
    groups=dfs_merged["bio_rep"])
mdf = model.fit()
#print(mdf.summary())

In [ ]:
residuals = mdf.resid
fitted_values = mdf.fittedvalues

In [ ]:
# Residuals plot
import statsmodels.api as sm
sm.qqplot(residuals, line='s')
plt.title("QQ-Plot of Residuals")
plt.show()

In [ ]:
print(mdf.summary())

In [ ]:
dfs_merged.to_csv(os.path.join('Output','Areas_.csv'))

### Created cropped and masked images for skeleton analysis
Skeleton Analysis outlined in [skan_analysis](/skan_analysis.ipynb)

In [ ]:
na_50_masks_files = sorted(glob('50_mm/Masks/*.tif'))
na_500_masks_files = sorted(glob('500_mm/Masks/*.tif'))
na_50_masks = list(map(sk.io.imread,na_50_masks_files))
na_500_masks = list(map(sk.io.imread,na_500_masks_files))
na_50_img_files = sorted(glob('50_mm/TIFFs/*.tiff'))
na_500_img_files = sorted(glob('500_mm/TIFFs/*.tiff'))
na_50_imgs = list(map(imread,na_50_img_files))
na_500_imgs = list(map(imread,na_500_img_files))

In [ ]:
#mask and crop images by oocyte mask bounding box
file_names = na_50_img_files + na_500_img_files
imgs = na_50_imgs + na_500_imgs
masks = na_50_masks + na_500_masks

In [ ]:
save_path = "cropped_imgs"
for i in range(len(file_names)):
    name = os.path.basename(file_names[i])
    img = imgs[i]
    mask = masks[i]
    props = sk.measure.regionprops(mask)
    minc,minr,maxc,maxr = props[0].bbox
    masked_img = mask*img
    cropped_img = masked_img[minc:maxc,minr:maxr]
    sk.io.imsave(os.path.join(save_path,name[:-5]+"_crop.tif"),cropped_img)